# Study 869 — 52-Week-High Breakout Drift — the teardown

The per-horizon splits, the Newey-West spread *t* (lags scaled to the overlap), the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'fingerprint': '357fd262912f', 'h5_days': 3039, 'h5_breakouts': 14630, 'h5_spread': 7.21, 'h5_t_nw': 1.14, 'h5_t_1s': 1.7, 'h5_brk': 31.14, 'h5_rest': 23.93, 'h5_welch': 1.22, 'h5_hit': 0.515, 'h5_placebo_obs': 7.21, 'h5_placebo_mean': -0.044, 'h5_placebo_sd': 5.66, 'h5_placebo_p': 0.099, 'h5_era_early': -1.5, 'h5_era_early_t': -0.24, 'h5_era_late': 14.72, 'h5_era_late_t': 1.4, 'h5_t1_gross': 7.21, 'h5_t1_cost': 4.68, 'h5_t1_net': 2.52, 'h5_t1_t': 0.6, 'h5_t5_gross': 7.21, 'h5_t5_cost': 20.68, 'h5_t5_net': -13.48, 'h5_t5_t': -3.18, 'h20_days': 3025, 'h20_breakouts': 14584, 'h20_spread': 21.64, 'h20_t_nw': 1.2, 'h20_t_1s': 2.48, 'h20_brk': 131.09, 'h20_rest': 109.45, 'h20_welch': 1.75, 'h20_hit': 0.506, 'h20_placebo_obs': 21.64, 'h20_placebo_mean': -0.284, 'h20_placebo_sd': 15.522, 'h20_placebo_p': 0.083, 'h20_era_early': 10.34, 'h20_era_early_t': 0.57, 'h20_era_late': 31.47, 'h20_era_late_t': 1.06, 'h20_t1_gross': 21.64, 'h20_t1_cost': 6.74, 'h20_t1_net': 14.9, 'h20_t1_t': 1.71, 'h20_t5_gross': 21.64, 'h20_t5_cost': 22.74, 'h20_t5_net': -1.1, 'h20_t5_t': -0.13, 'null_mean_t': -0.07, 'null_sd_t': 1.46, 'null_fire': 1, 'planted_t': 9.62, 'planted_welch': 14.78}

## The headline — long-breakout / short-rest forward-return spread

Daily equal-weight breakout-minus-rest forward return; NW lags = 2×horizon (forward windows overlap and breakouts cluster).

In [2]:
for h in ('h5','h20'):
    lab = '5-day' if h=='h5' else '20-day'
    print(f"{lab:>7}: spread {R[h+'_spread']:+.2f} bps  NW t = {R[h+'_t_nw']:+.2f}  "
          f"one-sample t = {R[h+'_t_1s']:+.2f}  |  breakout {R[h+'_brk']:+.2f} vs "
          f"rest {R[h+'_rest']:+.2f} bps (Welch {R[h+'_welch']:+.2f})  hit {R[h+'_hit']:.3f}")

  5-day: spread +7.21 bps  NW t = +1.14  one-sample t = +1.70  |  breakout +31.14 vs rest +23.93 bps (Welch +1.22)  hit 0.515
 20-day: spread +21.64 bps  NW t = +1.20  one-sample t = +2.48  |  breakout +131.09 vs rest +109.45 bps (Welch +1.75)  hit 0.506


## Placebo — column-permute the forward returns (1,000 permutations)

In [3]:
for h in ('h5','h20'):
    lab = '5-day' if h=='h5' else '20-day'
    print(f"{lab:>7}: observed {R[h+'_placebo_obs']:+.2f} bps vs placebo mean "
          f"{R[h+'_placebo_mean']:+.3f} (sd {R[h+'_placebo_sd']:.3f}) -> p = {R[h+'_placebo_p']:.3f}")

  5-day: observed +7.21 bps vs placebo mean -0.044 (sd 5.660) -> p = 0.099
 20-day: observed +21.64 bps vs placebo mean -0.284 (sd 15.522) -> p = 0.083


## Robustness — two eras (split 2018-01-01)

The (weak) drift is entirely a 2018-2026 phenomenon — neither era clears |*t*| = 2.

In [4]:
for h in ('h5','h20'):
    lab = '5-day' if h=='h5' else '20-day'
    print(f"{lab:>7}: 2010-2017 {R[h+'_era_early']:+.2f} bps (NW t={R[h+'_era_early_t']:+.2f})  |  "
          f"2018-2026 {R[h+'_era_late']:+.2f} bps (NW t={R[h+'_era_late_t']:+.2f})")

  5-day: 2010-2017 -1.50 bps (NW t=-0.24)  |  2018-2026 +14.72 bps (NW t=+1.40)
 20-day: 2010-2017 +10.34 bps (NW t=+0.57)  |  2018-2026 +31.47 bps (NW t=+1.06)


## The timer — can you get paid for it?

2 sides × (in+out) one-way cost per event; short pays 50 bps/yr borrow.

In [5]:
for h in ('h5','h20'):
    lab = '5-day' if h=='h5' else '20-day'
    print(f"{lab:>7} @1bp : gross {R[h+'_t1_gross']:+.2f} -> net {R[h+'_t1_net']:+.2f} bps "
          f"(cost {R[h+'_t1_cost']:.2f}, t={R[h+'_t1_t']:+.2f})")
    print(f"{lab:>7} @5bps: gross {R[h+'_t5_gross']:+.2f} -> net {R[h+'_t5_net']:+.2f} bps "
          f"(cost {R[h+'_t5_cost']:.2f}, t={R[h+'_t5_t']:+.2f})")

  5-day @1bp : gross +7.21 -> net +2.52 bps (cost 4.68, t=+0.60)
  5-day @5bps: gross +7.21 -> net -13.48 bps (cost 20.68, t=-3.18)
 20-day @1bp : gross +21.64 -> net +14.90 bps (cost 6.74, t=+1.71)
 20-day @5bps: gross +21.64 -> net -1.10 bps (cost 22.74, t=-0.13)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted breakout drift.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from breakout_high import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=869+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0015, seed=869, n_assets=40, n_days=1500))
print(f"planted (edge=0.0015): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.17 (sd 1.49), |t|>=2 in 0/8
planted (edge=0.0015): NW t = +9.62, Welch t = +14.78


## Verdict

- **Signal — Weak.** A fresh 52-week-high breakout is followed by a **drift up**, not a fade (the sign matches breakout momentum): +7.21 bps over 5 days, +21.64 bps over 20 days, breakout book beats the rest on both. But the overlap-corrected **NW *t* is only +1.14 / +1.20**, the placebo p is 0.10–0.08, and it is **entirely a 2018-2026 phenomenon** (2010-2017 flat-to-negative). It fails the |*t*| ≥ 2 bar and does not hold across eras. The 20-seed synthetic control recovers a *planted* drift cleanly (*t* = +9.62, fires on 1/20 nulls), so the weak real drift is a feature of the tape, not an engine bug.
- **Tradability — Mirage.** Even at an optimistic 1 bp one-way the net edge is insignificant (*t* = +0.60 / +1.71); at a realistic 5 bps it goes negative (-13.48 / -1.10 bps). No cost leaves a paycheck.